# StatPitch
## Notebook 1 / 5 — Data Collection

---

**What this notebook does:**
1. Downloads free international football data straight from GitHub (no API key needed)
2. Explores what columns and tournaments are available
3. Filters to the matches that are most useful for training
4. Adds a few extra columns to make analysis easier
5. Prints World Cup stats to confirm the data looks right
6. Saves clean CSV files ready for the next notebook

> **Data source:** [martj42/international-results](https://github.com/martj42/international-results) — a community-maintained dataset of every international football match since 1872, updated regularly.

---
## Step 1 — Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/StatPitch'
os.makedirs(SAVE_DIR, exist_ok=True)
os.chdir(SAVE_DIR)

print(f'Working directory: {os.getcwd()}')

warnings.filterwarnings('ignore')

# Make matplotlib charts look nicer
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f8'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

print('Libraries loaded!')

---
## Step 2 — Download the data

We load directly from GitHub using a public raw URL — no download, no sign-up.

In [ ]:
URL = 'https://raw.githubusercontent.com/martj42/international_results/master/results.csv'

print('Downloading data from GitHub...')
df = pd.read_csv(URL)

# Convert date column to proper datetime
df['date'] = pd.to_datetime(df['date'])

print(f'Done! Dataset has {df.shape[0]:,} matches and {df.shape[1]} columns.')
print(f'Date range: {df["date"].min().date()}  to  {df["date"].max().date()}')

Let's take a quick look at the raw data:

In [ ]:
# Show the first 10 rows
df.head(10)

In [ ]:
# Check column types and see if there are any missing values
df.info()
print('\nMissing values per column:')
print(df.isnull().sum())

---
## Step 3 — Explore available tournaments

The dataset contains many different competitions. Let's see what we have.

In [ ]:
print('Top 25 tournaments by number of matches:\n')
print(df['tournament'].value_counts().head(25).to_string())

---
## Step 4 — Filter to relevant matches

We only keep:
- **FIFA World Cup** — the actual tournament we want to predict
- **FIFA World Cup qualification** — gives us ~2,000 extra matches with the same teams
- **Friendlies** — useful for tracking team form between tournaments
- **Major continental tournaments** (Euro, Copa America, etc.) — high-quality competitive matches

We also focus on **1994 onwards** — modern football (tactical era, consistent rules, reliable data).

In [ ]:
RELEVANT_TOURNAMENTS = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'Friendly',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa America',
    'Africa Cup of Nations',
    'AFC Asian Cup',
    'CONCACAF Championship',
    'Confederations Cup',
    'UEFA Nations League',
]

df_filtered = df[
    (df['tournament'].isin(RELEVANT_TOURNAMENTS)) &
    (df['date'] >= '1994-01-01')
].copy()

print(f'Filtered dataset: {len(df_filtered):,} matches')
print(f'\nMatches per tournament:')
print(df_filtered['tournament'].value_counts().to_string())

---
## Step 5 — Clean and enhance the data

We add a few extra columns that will be very useful later:
- `total_goals` — total goals in the match (for Over/Under market)
- `goal_diff` — home goals minus away goals
- `result` — who won (home_win / draw / away_win)
- `year`, `month` — to track trends over time
- `is_neutral` — whether the match was played on neutral ground

In [ ]:
df_clean = df_filtered.sort_values('date').reset_index(drop=True)

# --- New columns ---

df_clean['total_goals'] = df_clean['home_score'] + df_clean['away_score']
df_clean['goal_diff']   = df_clean['home_score'] - df_clean['away_score']
df_clean['year']        = df_clean['date'].dt.year
df_clean['month']       = df_clean['date'].dt.month
df_clean['is_neutral']  = df_clean['neutral'].astype(int)  # 1 = neutral ground

# Result from the home team perspective
def classify_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

df_clean['result'] = df_clean.apply(classify_result, axis=1)

# Both teams scored? (BTTS market)
df_clean['btts'] = ((df_clean['home_score'] > 0) & (df_clean['away_score'] > 0)).astype(int)

# Over 2.5 goals? (common market)
df_clean['over_2_5'] = (df_clean['total_goals'] > 2.5).astype(int)

print('Clean dataset columns:')
print(df_clean.dtypes)
print(f'\nTotal rows: {len(df_clean):,}')

In [ ]:
# Preview the clean data
df_clean.head(10)

---
## Step 6 — World Cup specific statistics

Let's verify the data looks right by running some statistics on actual World Cup matches.

In [ ]:
wc = df_clean[df_clean['tournament'] == 'FIFA World Cup'].copy()

print(f'World Cup matches available: {len(wc)}')
print()

print('Matches per World Cup edition:')
print(wc.groupby('year')['result'].count().to_string())
print()

# Result distribution
rates = wc['result'].value_counts(normalize=True) * 100
print('--- Result rates (1994-present) ---')
print(f'  Home team wins:  {rates.get("home_win", 0):.1f}%')
print(f'  Draws:           {rates.get("draw", 0):.1f}%')
print(f'  Away team wins:  {rates.get("away_win", 0):.1f}%')
print()

# Goals
print('--- Goals ---')
print(f'  Average total goals per match:  {wc["total_goals"].mean():.2f}')
print(f'  % matches Over 2.5 goals:       {wc["over_2_5"].mean()*100:.1f}%')
print(f'  % matches Both Teams Scored:    {wc["btts"].mean()*100:.1f}%')
print()

# Top 5 highest scoring matches
print('--- Top 5 highest-scoring WC matches ---')
top5 = wc.nlargest(5, 'total_goals')[['date','home_team','home_score','away_score','away_team','total_goals']]
for _, row in top5.iterrows():
    print(f'  {row["home_team"]:20s} {int(row["home_score"])}-{int(row["away_score"])}  {row["away_team"]}  ({row["date"].date()})')

---
## Step 7 — Visualizations

Three quick charts to visually confirm the data makes sense.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('World Cup Match Statistics (1994-present)', fontsize=14, fontweight='bold', y=1.02)

# --- Chart 1: Result distribution (pie chart) ---
result_counts = wc['result'].value_counts()
colors = ['#4CAF50', '#9E9E9E', '#2196F3']
labels = ['Home win', 'Draw', 'Away win']
ordered_counts = [result_counts.get('home_win', 0),
                  result_counts.get('draw', 0),
                  result_counts.get('away_win', 0)]
axes[0].pie(ordered_counts, labels=labels, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Match Results', fontsize=12, fontweight='bold')

# --- Chart 2: Goals distribution ---
axes[1].hist(wc['total_goals'], bins=range(0, 12), color='#534AB7',
             alpha=0.8, edgecolor='white', linewidth=0.8)
mean_goals = wc['total_goals'].mean()
axes[1].axvline(mean_goals, color='red', linestyle='--', linewidth=2,
                label=f'Average: {mean_goals:.1f}')
axes[1].set_title('Total Goals per Match', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Total Goals')
axes[1].set_ylabel('Number of Matches')
axes[1].legend(fontsize=11)

# --- Chart 3: Over/Under & BTTS rates ---
categories = ['Over 2.5', 'Under 2.5', 'BTTS Yes', 'BTTS No']
over25_pct  = wc['over_2_5'].mean() * 100
under25_pct = 100 - over25_pct
btts_yes    = wc['btts'].mean() * 100
btts_no     = 100 - btts_yes
values = [over25_pct, under25_pct, btts_yes, btts_no]
bar_colors = ['#FF6B35', '#1D9E75', '#FF6B35', '#1D9E75']
bars = axes[2].bar(categories, values, color=bar_colors, edgecolor='white')
axes[2].set_title('Market Rates (%)', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Percentage (%)')
axes[2].set_ylim(0, 100)
for bar, val in zip(bars, values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('wc_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as wc_data_overview.png')

---
## Step 8 — Save the clean files

We save three CSV files:
- `football_data_clean.csv` — all filtered matches, used to train the model
- `world_cup_matches.csv` — actual World Cup games only (for backtesting)
- `world_cup_qualification.csv` — qualification matches (extra training data)

In [ ]:
# Full training set
df_clean.to_csv('football_data_clean.csv', index=False)

# World Cup matches only
wc.to_csv('world_cup_matches.csv', index=False)

# Qualification matches
wc_qual = df_clean[df_clean['tournament'] == 'FIFA World Cup qualification']
wc_qual.to_csv('world_cup_qualification.csv', index=False)

print('Files saved successfully!')
print()
print(f'  football_data_clean.csv       {len(df_clean):>6,} matches  (full training set)')
print(f'  world_cup_matches.csv         {len(wc):>6,} matches  (WC games only)')
print(f'  world_cup_qualification.csv   {len(wc_qual):>6,} matches  (qualification games)')
print()
print('Up next: Notebook 02 — Feature Engineering')
print('We will build Elo ratings, rolling averages, and head-to-head stats.')

---
## Summary

| File | Contents |
|------|----------|
| `football_data_clean.csv` | All training data (1994 - present) |
| `world_cup_matches.csv` | Only actual WC games — for testing accuracy |
| `world_cup_qualification.csv` | Qualification matches — more WC-context data |

**What we learned about World Cup data:**
- Home wins happen more often than away wins or draws
- Average goals per game is around 2.5 — so Over/Under 2.5 is roughly 50/50
- Both Teams To Score happens in roughly 50-55% of matches

These baseline rates are our starting point. The model needs to do **better** than these averages.

---